In [10]:
import pandas as pd
import math
import numpy as np



In [11]:
hospital_data_s2 = pd.read_csv("../data/curated/hospital_data/hospital_data_curated_sa2.csv")
rental_data_distance = pd.read_csv("../data/curated/rental_data_distance_sf.csv")
rental_data_distance["coordinates"] = rental_data_distance["coordinates"].apply(eval)
hospital_data_s2["coordinate"] = hospital_data_s2["coordinate"].apply(eval)
hospital_data_s2

,formal name,emergency capable,location address,suburb,postcode,category,agency type,coordinate,sa2_code_2021,sa2_name_2021
0,albert road clinic,no,31-33 albert road,south melbourne,3205,private,private hospital,"(-37.8354573, 144.9705465)",206051512.0,South Melbourne
1,albert road clinic,no,31-33 albert road,south melbourne,3205,private,private hospital,"(-37.8354573, 144.9705465)",206051511.0,Port Melbourne Industrial
2,albury wodonga health,yes,69 vermont street,wodonga,3690,public,public hospital,"(-36.1320403, 146.8800037)",204031491.0,Baranduda - Leneva
3,albury wodonga health,yes,69 vermont street,wodonga,3690,public,public hospital,"(-36.1320403, 146.8800037)",204031073.0,West Wodonga
4,albury wodonga health,yes,69 vermont street,wodonga,3690,public,public hospital,"(-36.1320403, 146.8800037)",204031492.0,Wodonga
...,...,...,...,...,...,...,...,...,...,...
492,yarram and district health service,no,85 commercial road,yarram,3971,public,public hospital,"(-38.557082550000004, 146.67819733634434)",205051104.0,Yarram
493,yarram and district health service,no,85 commercial road,yarram,3971,public,public hospital,"(-38.557082550000004, 146.67819733634434)",205041094.0,Churchill
494,yarrawonga health,no,33 piper street,yarrawonga,3730,public,public hospital,"(-36.0110552, 146.0059269)",216021414.0,Yarrawonga
495,yarrawonga health,no,33 piper street,yarrawonga,3730,public,public hospital,"(-36.0110552, 146.0059269)",216021412.0,Moira


In [12]:
EARTH_RADIUS = 6371
def calculate_distance(coordinate1, coordinate2):
    """ Function calculates the haversine distance between two coordinates """
    # Convert latitude and longitude from degrees to radians
    lat1 = np.radians(coordinate1[0])
    lon1 = np.radians(coordinate1[1])
    lat2 = np.radians(coordinate2[0])
    lon2 = np.radians(coordinate2[1])

    # Haversine formula
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = math.sin(dlat / 2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    distance = EARTH_RADIUS * c
    
    return distance
    

In [13]:
def distance_closest_hospital(rental_data_coordinate, rental_data_sa2, new_col_names, hospital_data):
    """ Function finds the distance to the closest hospital from the rental property """

    distance_dict = {}

    # Find closest university by comparing with all hospitals in victoria
    for index, hospital in hospital_data.iterrows():
        distance_dict[hospital["formal name"]] = calculate_distance(rental_data_coordinate, hospital["coordinate"])
    if distance_dict:
        # return closest hopsital
        return pd.Series({new_col_names[0]: min(distance_dict.items(), key=lambda x: x[1])[0],
                        new_col_names[1]: min(distance_dict.items(), key=lambda x: x[1])[1]})

In [14]:
new_col_names = ["closest hospital", "distance to closest hospital"]
new_col_hospital = rental_data_distance.apply(lambda x: distance_closest_hospital(
    x["coordinates"], x["SA2_CODE21"], new_col_names, hospital_data_s2), axis=1)
rental_data_distance = pd.concat([rental_data_distance, new_col_hospital], axis=1)

In [15]:
rental_data_distance.to_csv("../data/curated/rental_data_distance_sf.csv")

In [16]:
rental_data = pd.read_csv("../data/curated/rental_data_sf.csv")
rental_data_distance = pd.read_csv("../data/curated/rental_data_distance_sf.csv")
print(len(rental_data_distance))
print(len(rental_data))

11808
11808
